# B4-T1 walkthrough (exploration only)

The runnable deliverable is `python main.py`; this notebook only *inspects* its outputs and shows how to use the trained models. Run `python main.py` first.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
OUT = ROOT / 'outputs'

import keras
import pandas as pd
from IPython.display import Image, Markdown, display

from src.data.preprocessing import prepare_data
from src.models.uncertainty import UncertaintyEstimator
from src.utils.config import PipelineConfig

## 1. Base vs best FAIR model (test set)

In [ ]:
display(Markdown((OUT / 'tables' / 'base_vs_fair_test.md').read_text(encoding='utf-8')))
summary = json.loads((OUT / 'run_summary.json').read_text(encoding='utf-8'))
summary['best_fair']

## 2. Accuracy vs fairness trade-off (validation)

In [ ]:
display(Image(OUT / 'figures' / 'pareto_fairness.png'))
pd.read_csv(OUT / 'tables' / 'candidates_val.csv', index_col=0).sort_values('val_fair_corr').head(10)

## 3. Convergence of every final training

In [ ]:
display(Image(OUT / 'figures' / 'loss_curves.png'))

## 4. Uncertainty

In [ ]:
for name in ('uncertainty_by_class', 'uncertainty_by_missing_ext_sources', 'error_model_calibration'):
    display(Image(OUT / 'figures' / f'{name}.png'))

## 5. Predict class + uncertainty with the saved models

In [ ]:
config = PipelineConfig()
data = prepare_data(config.data, config.seed)
estimator = UncertaintyEstimator(
    classifier=keras.models.load_model(OUT / 'models' / 'fair_model.keras'),
    error_model=keras.models.load_model(OUT / 'models' / 'error_model.keras'),
    mc_samples=config.uncertainty.mc_samples,
)
estimator.predict(data.test.X[:10])